# 05 - Evaluation: Models Evaluation Comparison


In this notebook, we perform a comparative analysis of all trained models using:
- Metric distribution analysis and comparison (plots and summary statistics)
- Statistical significance testing (Corrected Resampled t-test)


## Import libraries and set the paths

In [1]:
from __future__ import annotations

import IPython.display as ipd
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from myocardial_infarction_mortality.config import MODELS_DIR, FIGURES_DIR
from myocardial_infarction_mortality.evaluation.visual_evaluation import plot_model_distribution
from myocardial_infarction_mortality.evaluation.statistical_test_evaluation import compute_pairwise_corrected_resampled_ttest

2026-03-11 12:23:45.787 | INFO     | myocardial_infarction_mortality.config:<module>:16 - PROJ_ROOT path is: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Cost-Sensitive-Learning-For-Myocardial-Infarction-Mortality-Prediction


In [2]:
EXPERIMENT_NAME = "smoteenn_auto__mec_fp1_fn10"

In [3]:
models_results_path: Path = MODELS_DIR / EXPERIMENT_NAME
print(f"Loading results at path:\n\t{models_results_path}")

Loading results at path:
	/home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Cost-Sensitive-Learning-For-Myocardial-Infarction-Mortality-Prediction/models/smoteenn_auto__mec_fp1_fn10


In [4]:
FIGURES_MODELS_COMPARISON_DIR = FIGURES_DIR / "EV_models_comparison_evaluation"
FIGURES_MODELS_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_MODELS_COMPARISON_EXPERIMENT_DIR = FIGURES_MODELS_COMPARISON_DIR / EXPERIMENT_NAME
FIGURES_MODELS_COMPARISON_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_MULTIPLE_MODELS_EVALUATION_DIR = FIGURES_MODELS_COMPARISON_EXPERIMENT_DIR / "multiple_models_evaluation"
FIGURES_MULTIPLE_MODELS_EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
pd.set_option("display.max_columns", None)
plt.rcParams.update({"font.size": 16})
sns.set_style("whitegrid")
sns.set_palette("tab10")

## Data Loading and Basic Overview

In [6]:
files = list(models_results_path.glob("*/generalization_metrics_summary.csv"))
generalization_df = pd.DataFrame()

print(f"Found {len(files)} files. Loading...")

if files:
    # Read and Concatenate
    # We use a generator expression inside concat for memory efficiency
    generalization_df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

    print("Success! Combined dataframe shape:", generalization_df.shape)
else:
    print(f"No files found in {models_results_path.absolute()}")

Found 12 files. Loading...
Success! Combined dataframe shape: (1200, 25)


In [7]:
generalization_df

,experiment_name,iteration,fold,model,split,tp,tn,fp,fn,accuracy,precision,recall,f1,specificity,fpr,balanced_accuracy,geometric_mean,mcc,kappa,roc_auc,average_precision,average_cost,score_time,fold_size,selected_features_names
0,smoteenn_auto__mec_fp1_fn10,1,1,KNORAE,generalization,25,1,132,0,0.164557,0.159236,1.00,0.274725,0.007519,0.992481,0.503759,0.086711,0.034601,0.002392,0.758346,0.496322,0.835443,0.507053,158,"['num_log1p_standard_scaler__L_BLOOD', 'num_lo..."
1,smoteenn_auto__mec_fp1_fn10,1,2,KNORAE,generalization,26,1,131,0,0.170886,0.165605,1.00,0.284153,0.007576,0.992424,0.503788,0.087039,0.035420,0.002506,0.714598,0.416719,0.829114,0.421383,158,"['num_log1p_standard_scaler__L_BLOOD', 'num_lo..."
2,smoteenn_auto__mec_fp1_fn10,1,3,KNORAE,generalization,24,6,126,1,0.191083,0.160000,0.96,0.274286,0.045455,0.954545,0.502727,0.208893,0.009670,0.001802,0.746970,0.339229,0.866242,0.301965,157,"['num_log1p_standard_scaler__L_BLOOD', 'num_lo..."
3,smoteenn_auto__mec_fp1_fn10,1,4,KNORAE,generalization,25,5,127,0,0.191083,0.164474,1.00,0.282486,0.037879,0.962121,0.518939,0.194625,0.078931,0.012383,0.765303,0.371737,0.808917,0.450954,157,"['num_log1p_standard_scaler__L_BLOOD', 'num_lo..."
4,smoteenn_auto__mec_fp1_fn10,1,5,KNORAE,generalization,25,1,131,0,0.165605,0.160256,1.00,0.276243,0.007576,0.992424,0.503788,0.087039,0.034843,0.002425,0.582424,0.234758,0.834395,0.410332,157,"['num_log1p_standard_scaler__L_BLOOD', 'num_lo..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,smoteenn_auto__mec_fp1_fn10,10,6,METADES,generalization,25,0,132,0,0.159236,0.159236,1.00,0.274725,0.000000,1.000000,0.500000,0.000000,0.000000,0.000000,0.769697,0.433145,0.840764,0.152789,157,"['num_log1p_standard_scaler__L_BLOOD', 'num_lo..."
1196,smoteenn_auto__mec_fp1_fn10,10,7,METADES,generalization,25,0,132,0,0.159236,0.159236,1.00,0.274725,0.000000,1.000000,0.500000,0.000000,0.000000,0.000000,0.842424,0.565466,0.840764,0.424270,157,"['num_log1p_standard_scaler__L_BLOOD', 'num_lo..."
1197,smoteenn_auto__mec_fp1_fn10,10,8,METADES,generalization,25,0,132,0,0.159236,0.159236,1.00,0.274725,0.000000,1.000000,0.500000,0.000000,0.000000,0.000000,0.767576,0.452461,0.840764,0.218081,157,"['num_log1p_standard_scaler__L_BLOOD', 'num_lo..."
1198,smoteenn_auto__mec_fp1_fn10,10,9,METADES,generalization,25,0,132,0,0.159236,0.159236,1.00,0.274725,0.000000,1.000000,0.500000,0.000000,0.000000,0.000000,0.856364,0.630721,0.840764,0.676863,157,"['num_log1p_standard_scaler__L_BLOOD', 'num_lo..."


## Fix the metrics and model to evaluate and compare

In [8]:
metrics_to_analyze = [
    "average_cost",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
]
print(f"Selected metrics:\n\t{metrics_to_analyze}")

Selected metrics:
	['average_cost', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']


In [9]:
#models_to_analyze = ["APriori", "APosteriori", "KNORAE", "MLPClassifier", ]
models_to_analyze = generalization_df["model"].unique()
print(f"Selected models:\n\t{models_to_analyze}")

generalization_df = generalization_df[generalization_df["model"].isin(models_to_analyze)]
generalization_df

Selected models:
	['KNORAE' 'LogisticRegression' 'Exponential' 'DecisionTreeClassifier'
 'StackingClassifier' 'DESKL' 'XGBClassifier' 'MLA'
 'RandomForestClassifier' 'VotingClassifier' 'SGDClassifier' 'METADES']


,experiment_name,iteration,fold,model,split,tp,tn,fp,fn,accuracy,precision,recall,f1,specificity,fpr,balanced_accuracy,geometric_mean,mcc,kappa,roc_auc,average_precision,average_cost,score_time,fold_size,selected_features_names
0,smoteenn_auto__mec_fp1_fn10,1,1,KNORAE,generalization,25,1,132,0,0.164557,0.159236,1.00,0.274725,0.007519,0.992481,0.503759,0.086711,0.034601,0.002392,0.758346,0.496322,0.835443,0.507053,158,"['num_log1p_standard_scaler__L_BLOOD', 'num_lo..."
1,smoteenn_auto__mec_fp1_fn10,1,2,KNORAE,generalization,26,1,131,0,0.170886,0.165605,1.00,0.284153,0.007576,0.992424,0.503788,0.087039,0.035420,0.002506,0.714598,0.416719,0.829114,0.421383,158,"['num_log1p_standard_scaler__L_BLOOD', 'num_lo..."
2,smoteenn_auto__mec_fp1_fn10,1,3,KNORAE,generalization,24,6,126,1,0.191083,0.160000,0.96,0.274286,0.045455,0.954545,0.502727,0.208893,0.009670,0.001802,0.746970,0.339229,0.866242,0.301965,157,"['num_log1p_standard_scaler__L_BLOOD', 'num_lo..."
3,smoteenn_auto__mec_fp1_fn10,1,4,KNORAE,generalization,25,5,127,0,0.191083,0.164474,1.00,0.282486,0.037879,0.962121,0.518939,0.194625,0.078931,0.012383,0.765303,0.371737,0.808917,0.450954,157,"['num_log1p_standard_scaler__L_BLOOD', 'num_lo..."
4,smoteenn_auto__mec_fp1_fn10,1,5,KNORAE,generalization,25,1,131,0,0.165605,0.160256,1.00,0.276243,0.007576,0.992424,0.503788,0.087039,0.034843,0.002425,0.582424,0.234758,0.834395,0.410332,157,"['num_log1p_standard_scaler__L_BLOOD', 'num_lo..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,smoteenn_auto__mec_fp1_fn10,10,6,METADES,generalization,25,0,132,0,0.159236,0.159236,1.00,0.274725,0.000000,1.000000,0.500000,0.000000,0.000000,0.000000,0.769697,0.433145,0.840764,0.152789,157,"['num_log1p_standard_scaler__L_BLOOD', 'num_lo..."
1196,smoteenn_auto__mec_fp1_fn10,10,7,METADES,generalization,25,0,132,0,0.159236,0.159236,1.00,0.274725,0.000000,1.000000,0.500000,0.000000,0.000000,0.000000,0.842424,0.565466,0.840764,0.424270,157,"['num_log1p_standard_scaler__L_BLOOD', 'num_lo..."
1197,smoteenn_auto__mec_fp1_fn10,10,8,METADES,generalization,25,0,132,0,0.159236,0.159236,1.00,0.274725,0.000000,1.000000,0.500000,0.000000,0.000000,0.000000,0.767576,0.452461,0.840764,0.218081,157,"['num_log1p_standard_scaler__L_BLOOD', 'num_lo..."
1198,smoteenn_auto__mec_fp1_fn10,10,9,METADES,generalization,25,0,132,0,0.159236,0.159236,1.00,0.274725,0.000000,1.000000,0.500000,0.000000,0.000000,0.000000,0.856364,0.630721,0.840764,0.676863,157,"['num_log1p_standard_scaler__L_BLOOD', 'num_lo..."


## Model Comparison (Distribution Analysis)

### Objective
To compare the performance distribution of all models for a specific metric on a targeted data split (e.g., Generalization or Resubstitution).

### Methodology
We generate a **distribution plot** and a **statistical report** for the selected metric and split:
* **Scope:** Parametrizable (Default: **Generalization**).
* **Visualization:**
    * **Boxplot:** Displays the median, Interquartile Range (IQR), and spread.
    * **Strip Plot:** Overlays raw data points (100 folds) to visualize density.
* **Statistics:** A summary table (Mean, Std, Min, Max) is displayed and saved to quantify the visual results.
* **Sorting:** Models are sorted **Alphabetically** to ensure consistent ordering across different plots.
* **Scale:** Fixed to $0.0 - 1.0$ (or customized based on metric range).

### Interpretation Guide
* **Box Height:** Higher is better for standard metrics (e.g., F1, MCC). Lower is better for error metrics.
* **Box Size (IQR):** A short box implies high stability (low variance). A tall box implies the model behavior changes drastically depending on the data fold.
* **Split Comparison:** By generating this plot for both 'resubstitution' and 'generalization', one can visually confirm overfitting (if the Resubstitution box is high/tight while the Generalization box is low/wide).

In [10]:
# List to store individual metric reports
metric_reports = []

for metric in metrics_to_analyze:
    report = plot_model_distribution(df=generalization_df,
                                     metric_name=metric,
                                     save_path=FIGURES_MULTIPLE_MODELS_EVALUATION_DIR,
                                     split_name="generalization")

    if report is not None:
        metric_reports.append(report)

# --- Final Aggregation ---
if metric_reports:
    # Concatenate all reports along columns (axis=1)
    # Since all reports share the same Index (Model Name), they will align automatically
    final_summary_df: pd.DataFrame = pd.concat(metric_reports, axis=1)

    print(f"\n{'='*80}\nFINAL DISTRIBUTION SUMMARY\n{'='*80}")
    ipd.display(final_summary_df)

    final_csv_path = FIGURES_MULTIPLE_MODELS_EVALUATION_DIR / "distribution_boxplot_summary.csv"
    final_summary_df.to_csv(final_csv_path)
else:
    print("No reports were generated.")


DISTRIBUTION ANALYSIS: AVERAGE_COST (GENERALIZATION)


,average_cost_count,average_cost_mean,average_cost_std,average_cost_min,average_cost_25%,average_cost_50%,average_cost_75%,average_cost_max
model,,,,,,,,
DESKL,100.0,0.840333,0.001666,0.835443,0.840764,0.840764,0.840764,0.841772
DecisionTreeClassifier,100.0,0.824225,0.049763,0.560510,0.840764,0.840764,0.840764,0.904459
Exponential,100.0,0.840333,0.001666,0.835443,0.840764,0.840764,0.840764,0.841772
KNORAE,100.0,0.833074,0.037566,0.726115,0.815287,0.828025,0.840764,0.962025
LogisticRegression,100.0,0.662712,0.072013,0.474684,0.605096,0.662420,0.719745,0.815287
METADES,100.0,0.840333,0.001666,0.835443,0.840764,0.840764,0.840764,0.841772
MLA,100.0,0.828886,0.041875,0.683544,0.808917,0.834395,0.840764,0.949045
RandomForestClassifier,100.0,0.840333,0.001666,0.835443,0.840764,0.840764,0.840764,0.841772
SGDClassifier,100.0,0.676078,0.111913,0.401274,0.600974,0.698420,0.753185,0.987261



DISTRIBUTION ANALYSIS: ACCURACY (GENERALIZATION)


,accuracy_count,accuracy_mean,accuracy_std,accuracy_min,accuracy_25%,accuracy_50%,accuracy_75%,accuracy_max
model,,,,,,,,
DESKL,100.0,0.159667,0.001666,0.158228,0.159236,0.159236,0.159236,0.164557
DecisionTreeClassifier,100.0,0.188379,0.068864,0.158228,0.159236,0.159236,0.164557,0.439490
Exponential,100.0,0.159667,0.001666,0.158228,0.159236,0.159236,0.159236,0.164557
KNORAE,100.0,0.178946,0.020177,0.152866,0.165605,0.171975,0.190176,0.273885
LogisticRegression,100.0,0.381932,0.065935,0.197452,0.335443,0.384121,0.433121,0.535032
METADES,100.0,0.159667,0.001666,0.158228,0.159236,0.159236,0.159236,0.164557
MLA,100.0,0.194012,0.046705,0.152866,0.159236,0.175159,0.215533,0.388535
RandomForestClassifier,100.0,0.159667,0.001666,0.158228,0.159236,0.159236,0.159236,0.164557
SGDClassifier,100.0,0.434424,0.186919,0.184713,0.278481,0.343949,0.611465,0.808917



DISTRIBUTION ANALYSIS: PRECISION (GENERALIZATION)


...

,f1_count,f1_mean,f1_std,f1_min,f1_25%,f1_50%,f1_75%,f1_max
model,,,,,,,,
DESKL,100.0,0.275363,0.002469,0.273224,0.274725,0.274725,0.274725,0.282609
DecisionTreeClassifier,100.0,0.281576,0.016393,0.270588,0.274725,0.274725,0.279954,0.362319
Exponential,100.0,0.275363,0.002469,0.273224,0.274725,0.274725,0.274725,0.282609
KNORAE,100.0,0.278390,0.007186,0.256983,0.274725,0.277778,0.282486,0.304878
LogisticRegression,100.0,0.335074,0.023116,0.284091,0.318748,0.332230,0.352113,0.409449
METADES,100.0,0.275363,0.002469,0.273224,0.274725,0.274725,0.274725,0.282609
MLA,100.0,0.280901,0.010864,0.265060,0.274725,0.277108,0.284153,0.323944
RandomForestClassifier,100.0,0.275363,0.002469,0.273224,0.274725,0.274725,0.274725,0.282609
SGDClassifier,100.0,0.358913,0.070868,0.280899,0.304878,0.326797,0.396236,0.550000



DISTRIBUTION ANALYSIS: ROC_AUC (GENERALIZATION)


,roc_auc_count,roc_auc_mean,roc_auc_std,roc_auc_min,roc_auc_25%,roc_auc_50%,roc_auc_75%,roc_auc_max
model,,,,,,,,
DESKL,100.0,0.818108,0.041818,0.701818,0.788409,0.818333,0.845227,0.923030
DecisionTreeClassifier,100.0,0.644378,0.079827,0.487576,0.596496,0.646136,0.708965,0.884697
Exponential,100.0,0.804431,0.047471,0.705455,0.768561,0.796818,0.839974,0.922727
KNORAE,100.0,0.724678,0.055330,0.582424,0.687273,0.720454,0.758759,0.863030
LogisticRegression,100.0,0.830067,0.046736,0.691515,0.800833,0.832879,0.865259,0.912424
METADES,100.0,0.811077,0.045242,0.712424,0.777652,0.807576,0.846742,0.919697
MLA,100.0,0.691148,0.060982,0.514697,0.660455,0.694188,0.733598,0.805606
RandomForestClassifier,100.0,0.831051,0.036760,0.714744,0.809091,0.834825,0.856541,0.921818
SGDClassifier,100.0,0.789586,0.066499,0.622121,0.748902,0.796364,0.844583,0.901818



FINAL DISTRIBUTION SUMMARY


,average_cost_count,average_cost_mean,average_cost_std,average_cost_min,average_cost_25%,average_cost_50%,average_cost_75%,average_cost_max,accuracy_count,accuracy_mean,accuracy_std,accuracy_min,accuracy_25%,accuracy_50%,accuracy_75%,accuracy_max,precision_count,precision_mean,precision_std,precision_min,precision_25%,precision_50%,precision_75%,precision_max,recall_count,recall_mean,recall_std,recall_min,recall_25%,recall_50%,recall_75%,recall_max,f1_count,f1_mean,f1_std,f1_min,f1_25%,f1_50%,f1_75%,f1_max,roc_auc_count,roc_auc_mean,roc_auc_std,roc_auc_min,roc_auc_25%,roc_auc_50%,roc_auc_75%,roc_auc_max
model,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
DESKL,100.0,0.840333,0.001666,0.835443,0.840764,0.840764,0.840764,0.841772,100.0,0.159667,0.001666,0.158228,0.159236,0.159236,0.159236,0.164557,100.0,0.159667,0.001666,0.158228,0.159236,0.159236,0.159236,0.164557,100.0,1.000000,0.000000,1.000000,1.00,1.000000,1.000000,1.0,100.0,0.275363,0.002469,0.273224,0.274725,0.274725,0.274725,0.282609,100.0,0.818108,0.041818,0.701818,0.788409,0.818333,0.845227,0.923030
DecisionTreeClassifier,100.0,0.824225,0.049763,0.560510,0.840764,0.840764,0.840764,0.904459,100.0,0.188379,0.068864,0.158228,0.159236,0.159236,0.164557,0.439490,100.0,0.164283,0.011928,0.158228,0.159236,0.159236,0.163588,0.221239,100.0,0.991200,0.025792,0.880000,1.00,1.000000,1.000000,1.0,100.0,0.281576,0.016393,0.270588,0.274725,0.274725,0.279954,0.362319,100.0,0.644378,0.079827,0.487576,0.596496,0.646136,0.708965,0.884697
Exponential,100.0,0.840333,0.001666,0.835443,0.840764,0.840764,0.840764,0.841772,100.0,0.159667,0.001666,0.158228,0.159236,0.159236,0.159236,0.164557,100.0,0.159667,0.001666,0.158228,0.159236,0.159236,0.159236,0.164557,100.0,1.000000,0.000000,1.000000,1.00,1.000000,1.000000,1.0,100.0,0.275363,0.002469,0.273224,0.274725,0.274725,0.274725,0.282609,100.0,0.804431,0.047471,0.705455,0.768561,0.796818,0.839974,0.922727
KNORAE,100.0,0.833074,0.037566,0.726115,0.815287,0.828025,0.840764,0.962025,100.0,0.178946,0.020177,0.152866,0.165605,0.171975,0.190176,0.273885,100.0,0.161937,0.004523,0.149351,0.160000,0.161290,0.164474,0.179856,100.0,0.991631,0.021451,0.880000,1.00,1.000000,1.000000,1.0,100.0,0.278390,0.007186,0.256983,0.274725,0.277778,0.282486,0.304878,100.0,0.724678,0.055330,0.582424,0.687273,0.720454,0.758759,0.863030
LogisticRegression,100.0,0.662712,0.072013,0.474684,0.605096,0.662420,0.719745,0.815287,100.0,0.381932,0.065935,0.197452,0.335443,0.384121,0.433121,0.535032,100.0,0.202858,0.016925,0.165563,0.190478,0.202466,0.215517,0.257426,100.0,0.968938,0.037339,0.840000,0.96,0.980769,1.000000,1.0,100.0,0.335074,0.023116,0.284091,0.318748,0.332230,0.352113,0.409449,100.0,0.830067,0.046736,0.691515,0.800833,0.832879,0.865259,0.912424
METADES,100.0,0.840333,0.001666,0.835443,0.840764,0.840764,0.840764,0.841772,100.0,0.159667,0.001666,0.158228,0.159236,0.159236,0.159236,0.164557,100.0,0.159667,0.001666,0.158228,0.159236,0.159236,0.159236,0.164557,100.0,1.000000,0.000000,1.000000,1.00,1.000000,1.000000,1.0,100.0,0.275363,0.002469,0.273224,0.274725,0.274725,0.274725,0.282609,100.0,0.811077,0.045242,0.712424,0.777652,0.807576,0.846742,0.919697
MLA,100.0,0.828886,0.041875,0.683544,0.808917,0.834395,0.840764,0.949045,100.0,0.194012,0.046705,0.152866,0.159236,0.175159,0.215533,0.388535,100.0,0.163920,0.007672,0.153846,0.159236,0.161290,0.165605,0.196581,100.0,0.984062,0.029369,0.880000,0.96,1.000000,1.000000,1.0,100.0,0.280901,0.010864,0.265060,0.274725,0.277108,0.284153,0.323944,100.0,0.691148,0.060982,0.514697,0.660455,0.694188,0.733598,0.805606
RandomForestClassifier,100.0,0.840333,0.001666,0.835443,0.840764,0.840764,0.840764,0.841772,100.0,0.159667,0.001666,0.158228,0.159236,0.159236,0.159236,0.164557,100.0,0.159667,0.001666,0.158228,0.159236,0.159236,0.159236,0.164557,100.0,1.000000,0.000000,1.000000,1.00,1.000000,1.000000,1.0,100.0,0.275363,0.002469,0.273224,0.274725,0.274725,0.274725,0.282609,100.0,0.831051,0.036760,0.714744,0.809091,0.834825,0.856541,0.

## Model Comparison & Statistical Significance Testing

### Objective
To definitively determine which models perform better than others by moving beyond simple average scores. We aim to identify **statistically significant differences** between model pairs, ensuring that observed performance gains are real and not merely artifacts of random data splitting.

### Methodology: Corrected Resampled t-test
Standard statistical tests (like the independent t-test) assume that data samples are independent. However, in Cross-Validation, training sets overlap significantly (e.g., 90% overlap in 10-fold CV), violating this assumption and leading to underestimated variance and high False Positive rates (Type I errors).

To address this, we employ the **Corrected Resampled t-test** proposed by Nadeau and Bengio (2003). This test adjusts the variance to account for the correlation between training sets and the ratio of testing to training samples.

The corrected t-statistic is calculated as:

$$t = \frac{\mu_{diff}}{\sqrt{\sigma_{diff}^2 \cdot (\frac{1}{n} + \frac{n_{test}}{n_{train}})}}$$

Where:
* $\mu_{diff}$: Mean difference in scores between Model A and Model B.
* $\sigma_{diff}^2$: Variance of the differences.
* $n$: Number of evaluations (Iterations $\times$ Folds).
* $\frac{n_{test}}{n_{train}}$: Correction factor for the overlap between training sets.

### Visualization: Pairwise Significance Heatmap
We visualize the results using a **Significance Heatmap** for each metric. This matrix displays the outcome of the hypothesis test for every pair of models based on the p-value.

* **Rows & Columns:** Each cell represents the comparison between the model on the row and the model on the column.
* **Color Coding:**
    * **Blue Cells ($p < 0.05$):** Indicate a **statistically significant difference**. Darker blues represent higher confidence (lower p-values, e.g., $p < 0.001$).
        * *Note:* This color confirms the models are *different*, but you must check the mean scores (or the CSV `Mean_Diff`) to see which one is *better*.
    * **Orange/Red Cells (NS):** Indicate **No Significant Difference**. The models perform statistically similarly, meaning any difference in their average scores is likely due to chance.

### Interpretation Guide
1.  **Identify "Cliques":** Look for blocks of **Orange (NS)** cells. These represent groups of models that are statistically indistinguishable from each other.
2.  **Verify Improvements:** If a high-scoring model has a **Blue** cell when compared to a baseline model, the improvement is real and statistically significant.
3.  **Symmetry:** The chart is symmetric. A Blue cell at (Row A, Col B) means A and B are significantly different. Use the `df_comparisons` table to confirm if A > B or B > A.

In [11]:
pairwise_frames = []

for metric in metrics_to_analyze:
    report = compute_pairwise_corrected_resampled_ttest(
        df=generalization_df,
        metric_name=metric, # Pass single string here
        n_train=0.9,
        n_test=0.1,
        save_path=FIGURES_MULTIPLE_MODELS_EVALUATION_DIR
    )

    if report is not None:
        pairwise_frames.append(report)

# --- Aggregate & Save Master CSV ---
if pairwise_frames:
    # Concatenate all metric-specific dataframes vertically (stacking rows)
    final_comparisons_df: pd.DataFrame = pd.concat(pairwise_frames, axis=0)

    print(f"\n{'='*80}\nFINAL CORRECTED RESAMPLED t-TEST SUMMARY\n{'='*80}")
    ipd.display(final_comparisons_df)

    # Define filename
    final_csv_path = FIGURES_MULTIPLE_MODELS_EVALUATION_DIR / "corrected_resampled_ttest_summary.csv"
    final_comparisons_df.to_csv(final_csv_path, index=False)
else:
    print("No pairwise comparisons were generated.")


CORRECTED RESAMPLED t-TEST ANALYSIS (Pairwise): AVERAGE_COST
Comparing 'DESKL' (mean=0.840) vs 'DecisionTreeClassifier' (mean=0.824)
  p-value: 0.3564 (t=0.9266) -> NOT significant
--------------------------------------------------------------------------------
Comparing 'DESKL' (mean=0.840) vs 'Exponential' (mean=0.840)
  p-value: 1.0000 (t=0.0000) -> NOT significant
--------------------------------------------------------------------------------
Comparing 'DESKL' (mean=0.840) vs 'KNORAE' (mean=0.833)
  p-value: 0.5817 (t=0.5527) -> NOT significant
--------------------------------------------------------------------------------
Comparing 'DESKL' (mean=0.840) vs 'LogisticRegression' (mean=0.663)
  p-value: 0.0000 (t=7.0562) -> Statistically SIGNIFICANT
--------------------------------------------------------------------------------
Comparing 'DESKL' (mean=0.840) vs 'METADES' (mean=0.840)
  p-value: 1.0000 (t=0.0000) -> NOT significant
--------------------------------------------------

,Metric,Model_A,Model_B,Mean_Diff,t-stat,p-value,is_significant,result
0,average_cost,DESKL,DecisionTreeClassifier,1.610780e-02,0.926599,3.563883e-01,False,No Diff
1,average_cost,DESKL,Exponential,-1.110223e-16,0.000000,1.000000e+00,False,No Diff
2,average_cost,DESKL,KNORAE,7.259131e-03,0.552731,5.816937e-01,False,No Diff
3,average_cost,DESKL,LogisticRegression,1.776207e-01,7.056215,2.366381e-10,True,Better
4,average_cost,DESKL,METADES,-1.110223e-16,0.000000,1.000000e+00,False,No Diff
...,...,...,...,...,...,...,...,...
127,roc_auc,XGBClassifier,MLA,1.666495e-01,6.855187,6.165310e-10,True,Better
128,roc_auc,XGBClassifier,RandomForestClassifier,2.674660e-02,1.651933,1.017166e-01,False,No Diff
129,roc_auc,XGBClassifier,SGDClassifier,6.821181e-02,2.406059,1.798128e-02,True,Better
130,roc_auc,XGBClassifier,StackingClassifier,1.333462e-03,0.072011,9.427383e-01,False,No Diff
